[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/08_rmsnorm.ipynb)

# 🟡 Medium: Implement RMSNorm

Implement **Root Mean Square Layer Normalization** — the normalization used in LLaMA, Gemma, etc.

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot w, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}$$

### Signature
```python
def rms_norm(x: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    # Normalize over the last dimension. No mean subtraction (unlike LayerNorm).
```

### Rules
- Do **NOT** use any built-in norm layers
- Normalize over `dim=-1`
- Must support autograd

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

# accelerate normalization removing centering
# norm over the last dim. in Transformer, the data is basically shaped (BS, len_seq, dim_embedding)
# so rms norm isn't just norm. x / rms already finishes normalization. then weight scales each normalized feature
    # decouples the strength of feature presence with feature importance
    # x / rms normalizes the embedding length induced by covariate shift.
    # weight then learns TRUE feature importance

def rms_norm(x, weight, eps=1e-6):
    # weight is for weighting each feature over a single sample
    x_dim = x.size()[-1]
    rms = torch.sqrt(torch.sum(x * x, dim=-1, keepdim=True) / x_dim + eps) # (..., 1)
    return x * weight / rms

    # rmsinv = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps)
    # return x * rmsinv * w

# element wise mult: 
    # a * b 
    # torch.mul(a, b)
# matrix dot product:
    # a @ b.mT (..., dim0, dim1) @ (..., dim1, dim2) => (..., dim0, dim2)
    # a @ b.T for mult w only two dims
    # torch.dot(a, b) 1d array only
    # torch.mm(a, b) 2d matrix only
    # torch.matmul(a, b) higher dims
# cross product:
    # torch.cross(a, b)

# brodcast
    # 1. right aligh
    # 2. unsqueeze missing dims
    # 3. from right to left: broadcast @ dim mismatch and one of them is 1
    # (a, b) & (a, 1) O
    # (a, b) & (a, 2) X
    # (a, b) & (1, ) O
    # (a, b) & (b, ) O
    # (a, b) & (a, ) X


In [32]:
# 🧪 Debug
x = torch.randn(2, 8)
w = torch.ones(8)
out = rms_norm(x, w)
print("Output shape:", out.shape)
print("RMS of output:", out.pow(2).mean(dim=-1).sqrt())  # should be ~1

torch.Size([2, 1])
Output shape: torch.Size([2, 8])
RMS of output: tensor([1.0000, 1.0000])


In [33]:
from torch_judge import check
check('rmsnorm')


🧪 Testing: Implement RMSNorm (Medium)
──────────────────────────────────────────────────
torch.Size([2, 1])
  ✅ [1/4] Basic behavior (1.3ms)
torch.Size([4, 1])
  ✅ [2/4] With learned weight (1.3ms)
torch.Size([2, 4, 1])
  ✅ [3/4] 3-D input (0.6ms)
torch.Size([2, 1])
  ✅ [4/4] Gradient flow (1.4ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (4.7ms total)
  Progress saved. Run status() to see your dashboard.

